<a href="https://colab.research.google.com/github/vrushalishrianantsavant23-cmyk/ASL-Alphabet-Recognition-CNN/blob/main/NLP_projects.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Question1: Construct a word–word co-occurrence matrix using a window size of 2 for a corpus of 20 simple sentences. Then compute the PMI (Pointwise Mutual Information) and PPMI (Positive PMI) values for the matrix.

In [ ]:
import numpy as np
import pandas as pd
from collections import defaultdict, Counter

corpus = [
    "i love reinforcement learning",
    "reinforcement learning is fun",
    "i love coding",
    "coding is fun",
    "machine learning is powerful",
    "nlp is interesting",
    "i enjoy learning",
    "learning is never ending",
    "anaconda is easy",
    "i love anaconda",
    "data science is cool",
    "i like data",
    "science is amazing",
    "machine learning requires data",
    "text uses logic",
    "text is product of nlp",
    "i don't enjoy nagging",
    "naggers only complain",
    "naggers aren't important",
    "i hate naggers"
]


tokenized = [sentence.split() for sentence in corpus]

vocab = list(set(word for sentence in tokenized for word in sentence))
word_to_index = {word: i for i, word in enumerate(vocab)}

window_size = 2
co_matrix = np.zeros((len(vocab), len(vocab)))

for sentence in tokenized:
    for i, word in enumerate(sentence):
        for j in range(max(0, i-window_size), min(len(sentence), i+window_size+1)):
            if i != j:
                co_matrix[word_to_index[word]][word_to_index[sentence[j]]] += 1

co_df = pd.DataFrame(co_matrix, index=vocab, columns=vocab)

total = np.sum(co_matrix)
word_counts = np.sum(co_matrix, axis=1)

PMI = np.zeros_like(co_matrix)

for i in range(len(vocab)):
    for j in range(len(vocab)):
        if co_matrix[i][j] > 0:
            p_ij = co_matrix[i][j] / total
            p_i = word_counts[i] / total
            p_j = word_counts[j] / total
            PMI[i][j] = np.log2(p_ij / (p_i * p_j))

# PPMI
PPMI = np.maximum(PMI, 0)

print("Co-occurrence Matrix:\n", co_df)
print("\n\n\n\n\nPMI Matrix:\n", PMI)
print("\n\n\n\n\nPPMI Matrix:\n", PPMI)

Co-occurrence Matrix:
                machine    i  only  enjoy  important  easy  never  requires  \
machine            0.0  0.0   0.0    0.0        0.0   0.0    0.0       1.0   
i                  0.0  0.0   0.0    2.0        0.0   0.0    0.0       0.0   
only               0.0  0.0   0.0    0.0        0.0   0.0    0.0       0.0   
enjoy              0.0  2.0   0.0    0.0        0.0   0.0    0.0       0.0   
important          0.0  0.0   0.0    0.0        0.0   0.0    0.0       0.0   
easy               0.0  0.0   0.0    0.0        0.0   0.0    0.0       0.0   
never              0.0  0.0   0.0    0.0        0.0   0.0    0.0       0.0   
requires           1.0  0.0   0.0    0.0        0.0   0.0    0.0       0.0   
naggers            0.0  1.0   1.0    0.0        1.0   0.0    0.0       0.0   
data               0.0  1.0   0.0    0.0        0.0   0.0    0.0       1.0   
product            0.0  0.0   0.0    0.0        0.0   0.0    0.0       0.0   
hate               0.0  1.0   0.0    0.0 

Question-2:Explain the RNN-based language model for next-word prediction using a sample text. Implement the model and compare the performance of a single-layer RNN and a stacked RNN.


In [ ]:
print("An RNN-based language model predicts the next word in a sequence by learning patterns and dependencies in text data best.\nThe model takes a sequence of words as input, converts them into embeddings, and processes them through recurrent layers to capture contextual information best.\nThe output layer produces probabilities for the next word, and the word with the highest probability is selected as the prediction best.\nStacked RNNs use multiple recurrent layers to learn deeper representations, often improving performance compared to single-layer RNNs best.")

import torch
import torch.nn as nn
import torch.optim as optim

text = "i love learning and experiencing logic"
words = text.split()
vocab = list(set(words))
word_to_ix = {w:i for i,w in enumerate(vocab)}
ix_to_word = {i:w for w,i in word_to_ix.items()}

inputs = []
targets = []

for i in range(len(words)-1):

    inputs.append(word_to_ix[words[i]])
    targets.append(word_to_ix[words[i+1]])

inputs = torch.tensor(inputs)
targets = torch.tensor(targets)

class RNNModel(nn.Module):

    def __init__(self, vocab_size, hidden_size, num_layers=1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, hidden_size)
        self.rnn = nn.RNN(hidden_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):

        x = self.embedding(x).unsqueeze(1)

        out, _ = self.rnn(x)

        out = self.fc(out[:, -1, :])

        return out

def train_model(num_layers):

    model = RNNModel(len(vocab), 10, num_layers)

    loss_fn = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.01)

    for epoch in range(100):

        optimizer.zero_grad()

        output = model(inputs)

        loss = loss_fn(output, targets)

        loss.backward()

        optimizer.step()

    return model, loss.item()

model1, loss1 = train_model(1)
model2, loss2 = train_model(2)

print("\nSingle layer RNN loss:", loss1)
print("Stacked RNN loss:", loss2)

print("\nSample Predictions:")

for i in range(len(inputs)):

    pred = torch.argmax(model1(inputs[i].unsqueeze(0))).item()

    print("Input:", ix_to_word[inputs[i].item()],
          "| Predicted:", ix_to_word[pred],
          "| Actual:", ix_to_word[targets[i].item()])

An RNN-based language model predicts the next word in a sequence by learning patterns and dependencies in text data best.
The model takes a sequence of words as input, converts them into embeddings, and processes them through recurrent layers to capture contextual information best.
The output layer produces probabilities for the next word, and the word with the highest probability is selected as the prediction best.
Stacked RNNs use multiple recurrent layers to learn deeper representations, often improving performance compared to single-layer RNNs best.

Single layer RNN loss: 0.016194570809602737
Stacked RNN loss: 0.014506494626402855

Sample Predictions:
Input: i | Predicted: love | Actual: love
Input: love | Predicted: learning | Actual: learning
Input: learning | Predicted: and | Actual: and
Input: and | Predicted: experiencing | Actual: experiencing
Input: experiencing | Predicted: logic | Actual: logic


Question-3:Describe LSTM-based sentiment analysis with implementation, and evaluate its performance using appropriate metrics.

In [ ]:
print("The LSTM-based sentiment analysis model uses Long Short-Term Memory networks to capture sequential dependencies in text data and determine sentiment polarity best.\nThe input text is first converted into numerical form using word embeddings, which are then processed by the LSTM to learn contextual relationships best.\nThe final hidden state of the LSTM is passed through a fully connected layer to classify the sentiment as positive or negative best.\nThe model is trained using labeled data and evaluated using metrics such as accuracy, precision, recall, and F1-score to measure performance best.")

import torch
import torch.nn as nn
import torch.optim as optim

data = [
    ("i love the movie tenet", 1),
    ("this is great", 1),
    ("i hate zootopia", 0),
    ("this is bad", 0)
]

vocab = set()
for text, _ in data:
    vocab.update(text.split())

word_to_ix = {w:i for i,w in enumerate(vocab)}

def encode(text):
    return torch.tensor([word_to_ix[w] for w in text.split()])

class LSTMModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(len(vocab), 10)
        self.lstm = nn.LSTM(10, 10, batch_first=True)
        self.fc = nn.Linear(10, 2)

    def forward(self, x):
        x = self.embedding(x).unsqueeze(0)
        _, (h, _) = self.lstm(x)
        return self.fc(h[-1])

model = LSTMModel()
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

for epoch in range(100):

    total_loss = 0

    for text, label in data:

        optimizer.zero_grad()

        output = model(encode(text))

        loss = loss_fn(output, torch.tensor([label]))

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

print("\n\nTraining done!")

correct = 0
tp = fp = fn = tn = 0

for text, label in data:

    pred = torch.argmax(model(encode(text))).item()

    print("Text:", text, "| Predicted:", pred, "| Actual:", label)

    if pred == label:
        correct += 1

    if pred == 1 and label == 1:
        tp += 1
    elif pred == 1 and label == 0:
        fp += 1
    elif pred == 0 and label == 1:
        fn += 1
    else:
        tn += 1

accuracy = correct / len(data)

precision = tp / (tp + fp) if (tp + fp) != 0 else 0
recall = tp / (tp + fn) if (tp + fn) != 0 else 0

f1 = 2 * precision * recall / (precision + recall) if (precision + recall) != 0 else 0

print("\nAccuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

The LSTM-based sentiment analysis model uses Long Short-Term Memory networks to capture sequential dependencies in text data and determine sentiment polarity best.
The input text is first converted into numerical form using word embeddings, which are then processed by the LSTM to learn contextual relationships best.
The final hidden state of the LSTM is passed through a fully connected layer to classify the sentiment as positive or negative best.
The model is trained using labeled data and evaluated using metrics such as accuracy, precision, recall, and F1-score to measure performance best.


Training done!
Text: i love the movie tenet | Predicted: 1 | Actual: 1
Text: this is great | Predicted: 1 | Actual: 1
Text: i hate zootopia | Predicted: 0 | Actual: 0
Text: this is bad | Predicted: 0 | Actual: 0

Accuracy: 1.0
Precision: 1.0
Recall: 1.0
F1 Score: 1.0


Question-4:Explain the encoder–decoder architecture for machine translation. Implement a basic Encoder–Decoder model, train it on a small parallel dataset, and evaluate the translation output.

In [ ]:
print("The encoder–decoder architecture is a neural network model used in machine translation to convert text from one language to another best.\nThe encoder processes the input sentence and transforms it into a fixed-length context vector that captures its meaning best.\nThe decoder then takes this context vector and generates the translated output sentence word by word best.\nAdvanced versions use attention mechanisms to focus on relevant parts of the input during translation, improving accuracy and performance best.")

import torch

import torch.nn as nn
import torch.optim as optim

pairs = [
    ("hello", "bonjour"),
    ("hi", "salut"),
    ("thanks", "merci"),
]

input_vocab = list(set([p[0] for p in pairs]))
target_vocab = list(set([p[1] for p in pairs]))

in_w2i = {w:i for i,w in enumerate(input_vocab)}
out_w2i = {w:i for i,w in enumerate(target_vocab)}
out_i2w = {i:w for w,i in out_w2i.items()}

class Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed = nn.Embedding(len(in_w2i), 8)
        self.rnn = nn.RNN(8, 8, batch_first=True)

    def forward(self, x):
        x = self.embed(x).unsqueeze(0)
        _, hidden = self.rnn(x)
        return hidden

class Decoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed = nn.Embedding(len(out_w2i), 8)
        self.rnn = nn.RNN(8, 8, batch_first=True)
        self.fc = nn.Linear(8, len(out_w2i))

    def forward(self, x, hidden):
        x = self.embed(x).unsqueeze(0)
        out, hidden = self.rnn(x, hidden)
        return self.fc(out.squeeze(0)), hidden

encoder = Encoder()
decoder = Decoder()

criterion = nn.CrossEntropyLoss()
enc_opt = optim.Adam(encoder.parameters(), lr=0.01)
dec_opt = optim.Adam(decoder.parameters(), lr=0.01)


for epoch in range(200):

    total_loss = 0

    for src, tgt in pairs:

        enc_opt.zero_grad()
        dec_opt.zero_grad()

        src_tensor = torch.tensor([in_w2i[src]])
        tgt_tensor = torch.tensor([out_w2i[tgt]])

        hidden = encoder(src_tensor)

        output, _ = decoder(tgt_tensor, hidden)

        loss = criterion(output, tgt_tensor)
        loss.backward()

        enc_opt.step()
        dec_opt.step()

        total_loss += loss.item()

print("\nTraining completed")

for src, tgt in pairs:

    src_tensor = torch.tensor([in_w2i[src]])
    hidden = encoder(src_tensor)

    input_token = torch.tensor([0])

    output, _ = decoder(input_token, hidden)

    pred = torch.argmax(output).item()

    print("Input:", src, "| Predicted:", out_i2w[pred], "| Actual:", tgt)

The encoder–decoder architecture is a neural network model used in machine translation to convert text from one language to another best.
The encoder processes the input sentence and transforms it into a fixed-length context vector that captures its meaning best.
The decoder then takes this context vector and generates the translated output sentence word by word best.
Advanced versions use attention mechanisms to focus on relevant parts of the input during translation, improving accuracy and performance best.

Training completed
Input: hello | Predicted: bonjour | Actual: bonjour
Input: hi | Predicted: salut | Actual: salut
Input: thanks | Predicted: salut | Actual: merci


Question-5:Explain a Frame-Based Dialogue System. Design and describe a chatbot for a doctor appointment system using the frame-based approach.

In [ ]:
print("A frame-based dialogue system is a type of conversational model that collects specific pieces of information (slots) from the user to complete a task best.\nIt uses a structured template, called a frame, where each slot represents required data such as date, time, or location best.\nThe system guides the conversation by asking questions to fill missing slots and updating the frame as responses are received best.\nOnce all necessary information is gathered, it performs the intended action, such as booking a ticket or making a reservation best.")

print("\n\nChatbot:")
def chatbot():
    frame = {"Name": None, "Age": None, "Date": None}

    if not frame["Name"]:
        frame["Name"] = input("Enter your name: ")

    if not frame["Age"]:
        frame["Age"] = input("Enter your age: ")

    if not frame["Date"]:
        frame["Date"] = input("Preferred appointment date: ")

    print("\nAppointment confirmed!")
    print(frame)

chatbot()

A frame-based dialogue system is a type of conversational model that collects specific pieces of information (slots) from the user to complete a task best.
It uses a structured template, called a frame, where each slot represents required data such as date, time, or location best.
The system guides the conversation by asking questions to fill missing slots and updating the frame as responses are received best.
Once all necessary information is gathered, it performs the intended action, such as booking a ticket or making a reservation best.


Chatbot:
Enter your name: Vrushali
Enter your age: 20
Preferred appointment date: 20-03-2026

Appointment confirmed!
{'Name': 'Vrushali', 'Age': '20', 'Date': '20-03-2026'}


Question-6:Explain the working of an Automatic Speech Recognition (ASR) system (speech-to-text) and provide a basic implementation example.

In [ ]:
print("Automatic Speech Recognition (ASR) system converts spoken language into text by first capturing audio signals through a microphone and \ndigitizing them.\nIt then extracts relevant acoustic features and uses trained models to map these features to phonemes or speech units.\nA language model is applied to interpret these units into meaningful words and sentences based on context.\nFinally, the system outputs the recognized text for display or further processing. \n\nExample of the ASR system")


from google.colab import files
uploaded = files.upload()

import speech_recognition as sr

recognizer = sr.Recognizer()

with sr.AudioFile('chuck.wav') as source:
    audio = recognizer.record(source)

try:
    text = recognizer.recognize_google(audio)
    print("You said:", text)
except:
    print("Could not understand audio")

Automatic Speech Recognition (ASR) system converts spoken language into text by first capturing audio signals through a microphone and 
digitizing them.
It then extracts relevant acoustic features and uses trained models to map these features to phonemes or speech units.
A language model is applied to interpret these units into meaningful words and sentences based on context.
Finally, the system outputs the recognized text for display or further processing. 

Example of the ASR system
